In [ ]:
import os
import glob
import numpy as np
from PIL import Image
import random
from tqdm import tqdm
import warnings
import gc
import shutil

# 1. RESEARCH CONFIGURATION
warnings.filterwarnings("ignore")
Image.MAX_IMAGE_PIXELS = None
random.seed(42)
np.random.seed(42)

CONFIG = {
    "dataset_path": "/kaggle/input",
    "output_path": "/kaggle/working/deforestation_data_final",
    "tile_size": 256,
    "stride": 160,
    "ndvi_change_threshold": 0.15,
    "vegetation_threshold": 0.30,
    "cloud_brightness_threshold": 0.35, # Aggressive cloud filtering
    "min_rgb_std": 0.012,               # FIXED: Added missing key
    "min_deforestation_pixels": 25,
    "negative_sample_ratio": 0.10,
    "max_tiles_per_pair": 800,
    "max_disk_gb": 7.8,
}

# Cleanup and Setup
if os.path.exists(CONFIG["output_path"]):
    shutil.rmtree(CONFIG["output_path"])
for s in ["train", "val", "test"]: 
    os.makedirs(os.path.join(CONFIG["output_path"], s), exist_ok=True)

# 2. STEP-BY-STEP HELPERS
def find_band(safe_path, band):
    # Searches specifically for 10m resolution imagery
    pattern = os.path.join(safe_path, "**", f"*{band}*.jp2")
    files = [f for f in glob.glob(pattern, recursive=True) if "R10m" in f or "IMG_DATA" in f]
    return sorted(files, key=os.path.getsize, reverse=True)[0] if files else None

def load_band_norm(path):
    if not path: return None
    try:
        with Image.open(path) as img:
            # Normalize Sentinel-2 DN to Reflectance (0.0 - 1.0)
            return np.array(img).astype(np.float32) / 10000.0
    except Exception:
        return None


# 3. PAIRING & TRACKING LOGIC
all_safes = sorted(glob.glob(os.path.join(CONFIG["dataset_path"], "**", "*.SAFE"), recursive=True))
tiles_dict = {}

# Group temporal acquisitions by their Geographic Tile ID
for folder in all_safes:
    try:
        # Sentinel-2 naming convention: ..._TXXXXX_...
        tile_id = os.path.basename(folder).split('_')[5]
        if tile_id not in tiles_dict: tiles_dict[tile_id] = []
        tiles_dict[tile_id].append(folder)
    except: continue

tile_counter = {"train": 0, "val": 0, "test": 0}
current_gb = 0.0

print(f" PIPELINE START: Target 8GB / 16,000 Tiles")
print(f"Detected {len(tiles_dict)} unique geographic regions (Tile IDs).")


# 4. MAIN PROCESSING LOOP

for tile_id, folders in tiles_dict.items():
    print(f"\n[Region {tile_id}] - {len(folders)} time-steps found.")
    
    # Macro-level tracking for each geographic region
    pbar = tqdm(range(len(folders) - 1), desc=f"Processing {tile_id}", leave=True)
    
    for i in pbar:
        # Safety Stop: Prevents Kaggle Disk Full Error
        if current_gb >= CONFIG["max_disk_gb"]: 
            print("\n Disk limit approached. Stopping.")
            break
        
        path_before, path_after = folders[i], folders[i+1]
        
        try:
            # STEP 1: LOAD BANDS (Checking both dates)
            r_b = load_band_norm(find_band(path_before, "B04"))
            n_b = load_band_norm(find_band(path_before, "B08"))
            
            r_a = load_band_norm(find_band(path_after, "B04"))
            g_a = load_band_norm(find_band(path_after, "B03"))
            b_a = load_band_norm(find_band(path_after, "B02"))
            n_a = load_band_norm(find_band(path_after, "B08"))

            if any(v is None for v in [r_b, n_b, r_a, g_a, b_a, n_a]):
                continue

            # STEP 2: QUALITY & SHAPE CHECKS
            if r_b.shape != r_a.shape:
                continue
            
            # Simple Cloud Masking using Blue Band brightness
            if np.mean(b_a) > CONFIG["cloud_brightness_threshold"]:
                continue

            # STEP 3: MASKING (NDVI DIFFERENCING)
            # NDVI Formula: (NIR - Red) / (NIR + Red)
            ndvi_b = (n_b - r_b) / (n_b + r_b + 1e-6)
            ndvi_a = (n_a - r_a) / (n_a + r_a + 1e-6)
            
            # Change Mask: High NDVI previously AND high drop in NDVI currently
            mask = ((ndvi_b > CONFIG["vegetation_threshold"]) & 
                    ((ndvi_b - ndvi_a) > CONFIG["ndvi_change_threshold"])).astype(np.uint8)
            
            rgb = np.stack([r_a, g_a, b_a], axis=-1)
            
            # Cleanup intermediate arrays to free RAM
            del r_b, n_b, r_a, g_a, b_a, n_a, ndvi_b, ndvi_a
            gc.collect()

            # Assign Split at PAIR level to prevent spatial leakage
            rnd_split = random.random()
            split = "test" if rnd_split < 0.15 else "val" if rnd_split < 0.30 else "train"

            # STEP 4: RANDOMIZED TILING
            h, w = mask.shape
            coords = [(y, x) for y in range(0, h-256+1, 160) for x in range(0, w-256+1, 160)]
            random.shuffle(coords)

            pair_tiles_saved = 0
            for y, x in coords:
                if pair_tiles_saved >= CONFIG["max_tiles_per_pair"]: break
                
                m_t = mask[y:y+256, x:x+256]
                rgb_t = rgb[y:y+256, x:x+256]
                chg_px = np.count_nonzero(m_t)
                
                keep = False
                # Scenario A: Deforestation detected
                if chg_px >= CONFIG["min_deforestation_pixels"]:
                    keep = True
                # Scenario B: Negative sample (Healthy Forest)
                elif chg_px == 0 and random.random() < CONFIG["negative_sample_ratio"]:
                    # min_rgb_std ensures we don't save blank/black tiles
                    if np.std(rgb_t) > CONFIG["min_rgb_std"]:
                        keep = True

                if keep:
                    idx = tile_counter[split]
                    fname = f"{CONFIG['output_path']}/{split}/{idx}.npz"
                    
                    # Save as compressed float16 to save significant disk space
                    np.savez_compressed(fname, 
                                        image=rgb_t.astype(np.float16), 
                                        mask=m_t, 
                                        source=os.path.basename(path_after))
                    
                    # Update usage tracking (Incremental is faster than os.walk)
                    current_gb += os.path.getsize(fname) / (1024**3)
                    tile_counter[split] += 1
                    pair_tiles_saved += 1
            
            # Update Micro-Progress Bar with real-time stats
            pbar.set_postfix({
                'TotalTiles': sum(tile_counter.values()), 
                'DiskGB': f"{current_gb:.2f}"
            })
            
            del rgb, mask
            gc.collect()

        except Exception as e:
            tqdm.write(f"Error on Pair {i}: {e}")
            continue


print(f" PIPELINE COMPLETE")
print(f"Total Tiles: {sum(tile_counter.values())}")
print(f"Final Size: {current_gb:.2f} GB")
print(f"Breakdown: Train={tile_counter['train']}, Val={tile_counter['val']}, Test={tile_counter['test']}")
print(f"Output Directory: {CONFIG['output_path']}")